# Notebook 02 — V2: La brecha que no cierra (slope chart)

**Visualización:** Slope chart en Flourish  
**Pregunta:** ¿Ha mejorado la brecha de bienestar entre géneros en los últimos 10 años (2014→2024)?  
**Output:** `data/processed/v2_slope_temporal.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../')
OUT  = ROOT / 'data' / 'processed'

COUNTRY_NAMES = {
    'AT': 'Austria', 'BE': 'Belgium', 'BG': 'Bulgaria',
    'CH': 'Switzerland', 'CY': 'Cyprus', 'CZ': 'Czechia',
    'DE': 'Germany', 'DK': 'Denmark', 'EE': 'Estonia',
    'ES': 'Spain', 'FI': 'Finland', 'FR': 'France',
    'GB': 'United Kingdom', 'GR': 'Greece', 'HR': 'Croatia',
    'HU': 'Hungary', 'IE': 'Ireland', 'IL': 'Israel',
    'IS': 'Iceland', 'IT': 'Italy', 'LT': 'Lithuania',
    'LV': 'Latvia', 'ME': 'Montenegro', 'NL': 'Netherlands',
    'NO': 'Norway', 'PL': 'Poland', 'PT': 'Portugal',
    'RS': 'Serbia', 'SE': 'Sweden', 'SI': 'Slovenia',
    'SK': 'Slovakia', 'UA': 'Ukraine'
}

print('Setup OK')

Setup OK


## 1. Cargar datasets limpios

In [2]:
ess11 = pd.read_csv(OUT / 'ESS11_clean.csv', low_memory=False)
ess7  = pd.read_csv(OUT / 'ESS7_clean.csv',  low_memory=False)

print(f'ESS R11: {len(ess11):,} filas — países: {sorted(ess11["cntry"].unique())}')
print(f'ESS R7:  {len(ess7):,} filas  — países: {sorted(ess7["cntry"].unique())}')

paises_comunes = sorted(set(ess11['cntry'].unique()) & set(ess7['cntry'].unique()))
print(f'\nPaíses en ambas rondas ({len(paises_comunes)}): {paises_comunes}')

ESS R11: 50,116 filas — países: ['AT', 'BE', 'BG', 'CH', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IL', 'IS', 'IT', 'LT', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RS', 'SE', 'SI', 'SK', 'UA']
ESS R7:  40,185 filas  — países: ['AT', 'BE', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'IL', 'LT', 'NL', 'NO', 'PL', 'PT', 'SE', 'SI']

Países en ambas rondas (19): ['AT', 'BE', 'CH', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'IL', 'LT', 'NL', 'NO', 'PL', 'PT', 'SE', 'SI']


## 2. Calcular BRECHA_GNDR por ronda

In [3]:
def brecha_por_pais(df, suffix):
    """Calcula wb_hombres, wb_mujeres y BRECHA_GNDR por país para un dataset dado."""
    # Solo países con ambos géneros presentes
    wb = (
        df.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
        .agg(mean='mean', n='count')
        .reset_index()
    )
    wide = wb.pivot(index='cntry', columns='gndr', values=['mean', 'n'])
    # gndr=1 → hombres, gndr=2 → mujeres
    wide.columns = [f'wb_h_{suffix}', f'wb_m_{suffix}', f'n_h_{suffix}', f'n_m_{suffix}']
    wide = wide.reset_index()
    wide[f'brecha_{suffix}'] = (wide[f'wb_h_{suffix}'] - wide[f'wb_m_{suffix}']).round(4)
    wide[f'n_{suffix}'] = wide[f'n_h_{suffix}'] + wide[f'n_m_{suffix}']
    # Eliminar filas donde falte algún género
    wide = wide.dropna(subset=[f'brecha_{suffix}'])
    return wide

b_r7  = brecha_por_pais(ess7,  'r7')
b_r11 = brecha_por_pais(ess11, 'r11')

print(f'Países con brecha calculada en R7:  {len(b_r7)}')
print(f'Países con brecha calculada en R11: {len(b_r11)}')

Países con brecha calculada en R7:  21
Países con brecha calculada en R11: 30


## 3. Merge y cálculo del delta

In [4]:
v2 = b_r7[['cntry', 'brecha_r7', 'n_r7']].merge(
     b_r11[['cntry', 'brecha_r11', 'n_r11']], on='cntry', how='inner')

v2['delta'] = (v2['brecha_r11'] - v2['brecha_r7']).round(4)
v2['country_name'] = v2['cntry'].map(COUNTRY_NAMES)
v2['es_espana'] = v2['cntry'] == 'ES'

# Clasificar tendencia
def clasificar(delta):
    if delta < -0.02:
        return 'mejora'      # brecha se reduce
    elif delta > 0.02:
        return 'empeora'     # brecha crece
    else:
        return 'estable'

v2['grupo'] = v2['delta'].apply(clasificar)

print(f'Países en el slope chart: {len(v2)}')
print(f"\nGrupos: {v2['grupo'].value_counts().to_dict()}")
print()
print(v2.sort_values('delta').to_string(index=False))

Países en el slope chart: 19

Grupos: {'mejora': 9, 'estable': 7, 'empeora': 3}

cntry  brecha_r7   n_r7  brecha_r11  n_r11   delta   country_name  es_espana   grupo
   LT     0.1613 2238.0      0.0217 1363.0 -0.1396      Lithuania      False  mejora
   PL     0.2107 1611.0      0.1033 1429.0 -0.1074         Poland      False  mejora
   DE     0.1231 3040.0      0.0200 2419.0 -0.1031        Germany      False  mejora
   SE     0.1551 1789.0      0.0629 1230.0 -0.0922         Sweden      False  mejora
   SI     0.1721 1222.0      0.1075 1248.0 -0.0646       Slovenia      False  mejora
   IE     0.0054 2382.0     -0.0472 2012.0 -0.0526        Ireland      False  mejora
   IL     0.1436 2520.0      0.1001  904.0 -0.0435         Israel      False  mejora
   PT     0.3239 1264.0      0.2812 1373.0 -0.0427       Portugal      False  mejora
   EE     0.0652 2047.0      0.0390 1293.0 -0.0262        Estonia      False  mejora
   FI     0.0188 2083.0      0.0023 1562.0 -0.0165        Finland    

## 4. Exploración detallada

In [5]:
print('Estadísticos del delta (brecha_r11 - brecha_r7):')
print(v2['delta'].describe().round(4))
print()
print('Países donde la brecha MEJORÓ (se redujo):')
print(v2[v2['grupo']=='mejora'][['cntry','country_name','brecha_r7','brecha_r11','delta']]
      .sort_values('delta').to_string(index=False))
print()
print('Países donde la brecha EMPEORÓ (creció):')
print(v2[v2['grupo']=='empeora'][['cntry','country_name','brecha_r7','brecha_r11','delta']]
      .sort_values('delta', ascending=False).to_string(index=False))
print()
print('Países ESTABLES:')
print(v2[v2['grupo']=='estable'][['cntry','country_name','brecha_r7','brecha_r11','delta']]
      .to_string(index=False))
print()
print('España:')
print(v2[v2['cntry']=='ES'][['cntry','country_name','brecha_r7','brecha_r11','delta','grupo']]
      .to_string(index=False))

Estadísticos del delta (brecha_r11 - brecha_r7):
count    19.0000
mean     -0.0292
std       0.0538
min      -0.1396
25%      -0.0586
50%      -0.0165
75%       0.0070
max       0.0478
Name: delta, dtype: float64

Países donde la brecha MEJORÓ (se redujo):
cntry country_name  brecha_r7  brecha_r11   delta
   LT    Lithuania     0.1613      0.0217 -0.1396
   PL       Poland     0.2107      0.1033 -0.1074
   DE      Germany     0.1231      0.0200 -0.1031
   SE       Sweden     0.1551      0.0629 -0.0922
   SI     Slovenia     0.1721      0.1075 -0.0646
   IE      Ireland     0.0054     -0.0472 -0.0526
   IL       Israel     0.1436      0.1001 -0.0435
   PT     Portugal     0.3239      0.2812 -0.0427
   EE      Estonia     0.0652      0.0390 -0.0262

Países donde la brecha EMPEORÓ (creció):
cntry country_name  brecha_r7  brecha_r11  delta
   AT      Austria     0.0177      0.0655 0.0478
   NL  Netherlands     0.0772      0.1249 0.0477
   BE      Belgium     0.1360      0.1701 0.0341

País

## 5. Formato final para Flourish

El slope chart en Flourish necesita dos columnas numéricas (un punto por año) y una columna de etiqueta/color.

In [6]:
# Redondear para legibilidad en tooltips
v2['brecha_r7']  = v2['brecha_r7'].round(3)
v2['brecha_r11'] = v2['brecha_r11'].round(3)
v2['delta']      = v2['delta'].round(3)

# Seleccionar y ordenar columnas finales
v2_out = v2[['cntry', 'country_name', 'brecha_r7', 'brecha_r11',
             'delta', 'grupo', 'es_espana', 'n_r7', 'n_r11']].copy()

v2_out = v2_out.sort_values('brecha_r11', ascending=False).reset_index(drop=True)

print('Formato final:')
print(v2_out.to_string(index=False))

Formato final:
cntry   country_name  brecha_r7  brecha_r11  delta   grupo  es_espana   n_r7  n_r11
   PT       Portugal      0.324       0.281 -0.043  mejora      False 1264.0 1373.0
   ES          Spain      0.189       0.205  0.016 estable       True 1925.0 1844.0
   BE        Belgium      0.136       0.170  0.034 empeora      False 1769.0 1594.0
   FR         France      0.181       0.167 -0.014 estable      False 1916.0 1769.0
   NL    Netherlands      0.077       0.125  0.048 empeora      False 1917.0 1694.0
   SI       Slovenia      0.172       0.108 -0.065  mejora      False 1222.0 1248.0
   PL         Poland      0.211       0.103 -0.107  mejora      False 1611.0 1429.0
   IL         Israel      0.144       0.100 -0.044  mejora      False 2520.0  904.0
   HU        Hungary      0.087       0.100  0.012 estable      False 1696.0 2117.0
   AT        Austria      0.018       0.066  0.048 empeora      False 1795.0 2350.0
   SE         Sweden      0.155       0.063 -0.092  mejora   

## 6. Verificar y guardar

In [7]:
def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)}')
    print(f'Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos:\n{nulos_con}')
    else:
        print('Sin valores nulos')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    print(f'Muestra:\n{df.head(3).to_string()}')
    print('✓ OK')

verificar_csv_flourish(v2_out, 'v2_slope_temporal')


=== v2_slope_temporal ===
Filas: 19
Columnas: ['cntry', 'country_name', 'brecha_r7', 'brecha_r11', 'delta', 'grupo', 'es_espana', 'n_r7', 'n_r11']
Sin valores nulos
Muestra:
  cntry country_name  brecha_r7  brecha_r11  delta    grupo  es_espana    n_r7   n_r11
0    PT     Portugal      0.324       0.281 -0.043   mejora      False  1264.0  1373.0
1    ES        Spain      0.189       0.205  0.016  estable       True  1925.0  1844.0
2    BE      Belgium      0.136       0.170  0.034  empeora      False  1769.0  1594.0
✓ OK


In [8]:
out_path = OUT / 'v2_slope_temporal.csv'
v2_out.to_csv(out_path, index=False)
print(f'Guardado: {out_path}')

Guardado: ../data/processed/v2_slope_temporal.csv


## 7. Resumen

In [9]:
print('=' * 60)
print('RESUMEN V2 — SLOPE CHART TEMPORAL')
print('=' * 60)
print()
print(f'Países en el slope chart: {len(v2_out)}')
print(f'Rango brecha R7:  [{v2_out["brecha_r7"].min():.3f}, {v2_out["brecha_r7"].max():.3f}]')
print(f'Rango brecha R11: [{v2_out["brecha_r11"].min():.3f}, {v2_out["brecha_r11"].max():.3f}]')
print(f'Media delta: {v2_out["delta"].mean():.3f}')
print()
counts = v2_out['grupo'].value_counts()
print(f'Mejora  (delta < -0.02): {counts.get("mejora", 0)} países')
print(f'Estable (-0.02 a +0.02): {counts.get("estable", 0)} países')
print(f'Empeora (delta > +0.02): {counts.get("empeora", 0)} países')
print()
es = v2_out[v2_out['cntry']=='ES'].iloc[0]
print(f'España: brecha_r7={es.brecha_r7:.3f} → brecha_r11={es.brecha_r11:.3f}  delta={es.delta:+.3f}  ({es.grupo})')
print()
print('Configuración Flourish:')
print('  Tipo: Slope chart (Line chart con 2 puntos)')
print('  Columnas de valor: brecha_r7 (2014), brecha_r11 (2024)')
print('  Etiqueta: country_name')
print('  Color por: grupo (mejora=verde / estable=gris / empeora=rojo)')
print('  España: color/grosor diferenciado via es_espana')

RESUMEN V2 — SLOPE CHART TEMPORAL

Países en el slope chart: 19
Rango brecha R7:  [0.005, 0.324]
Rango brecha R11: [-0.047, 0.281]
Media delta: -0.029

Mejora  (delta < -0.02): 9 países
Estable (-0.02 a +0.02): 7 países
Empeora (delta > +0.02): 3 países

España: brecha_r7=0.189 → brecha_r11=0.205  delta=+0.016  (estable)

Configuración Flourish:
  Tipo: Slope chart (Line chart con 2 puntos)
  Columnas de valor: brecha_r7 (2014), brecha_r11 (2024)
  Etiqueta: country_name
  Color por: grupo (mejora=verde / estable=gris / empeora=rojo)
  España: color/grosor diferenciado via es_espana
